# Étape 1 — Environnement et baseline
Projet : agent de tri d'offres de stage/emploi (extraction structurée + matching CV)

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4.

In [ ]:
# 1. Installer les bibliothèques
!pip install -q transformers datasets accelerate bitsandbytes peft trl

In [ ]:
# 2. Se connecter à Hugging Face (colle ton token avec droits 'write')
from huggingface_hub import login
login()

In [ ]:
# 3. Vérifier que le GPU est actif
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 4. Charger le modèle de base
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)
print('Modèle chargé')

In [ ]:
# 5. Le schéma JSON cible (à figer avant d'annoter)
SCHEMA = '''{
  "poste": string,
  "entreprise": string | null,
  "ville": string | null,
  "type_contrat": "PFE" | "stage" | "CDI" | "CDD" | "freelance" | null,
  "duree_mois": number | null,
  "competences": [string],
  "remuneration": string | null,
  "langue_offre": "fr" | "en" | "ar"
}'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    "Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n" + SCHEMA
)

In [ ]:
# 6. Fonction d'extraction
import json

def extraire(offre: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': offre},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    n = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0][n:], skip_special_tokens=True)

def est_json_valide(texte: str) -> bool:
    try:
        json.loads(texte.strip().removeprefix('```json').removesuffix('```'))
        return True
    except json.JSONDecodeError:
        return False

In [ ]:
# 7. Tester sur des offres
# Les offres réelles utilisées pour la baseline ne sont pas publiées (droits des sites d'origine).
# Remplace l'exemple fictif ci-dessous par tes propres offres.
offres = {
    'exemple_FR': '''
Stage PFE - Développeur Full Stack (React / NestJS)
Société Exemple, Tunis. Durée : 6 mois à partir de février.
Compétences : TypeScript, React, NestJS, PostgreSQL, Docker.
Stage rémunéré.
''',
}

for nom, texte in offres.items():
    sortie = extraire(texte)
    print(f'# {nom} | valide = {est_json_valide(sortie)}')
    print(sortie, '\n')


In [ ]:
# 7b. Synthèse compacte des sorties brutes
import textwrap
for nom, s in resultats.items():
    print('#', nom, '| valide =', est_json_valide(s))
    print(textwrap.fill(' '.join(s.split()), 120))
    print()

## 8. Note tes observations (baseline)
Teste 5 à 10 offres différentes (FR, EN, AR) et note ici ce que le modèle rate :
- JSON invalide ?
- Champs inventés ?
- Compétences oubliées ?
- Mauvaise gestion de l'arabe ?

Ces erreurs justifient le fine-tuning de l'étape suivante.

In [ ]:
!pip install -q pydantic
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal["PFE", "stage", "CDI", "CDD", "freelance"]]
    mode_travail: Optional[Literal["présentiel", "hybride", "remote"]]
    duree_mois: Optional[float]
    competences: list[str]
    remuneration: Optional[str]
    langue_offre: Literal["fr", "en", "ar"]

def nettoyer(texte):
    return texte.strip().removeprefix("```json").removesuffix("```").strip()

def respecte_schema(texte):
    try:
        Offre.model_validate_json(nettoyer(texte))
        return True
    except ValidationError as e:
        print(e.errors()[0]["loc"], e.errors()[0]["msg"])
        return False